# Assignment 20: Content-Based Movie Recommendation System
**Author:** Parth Dadhaniya  
**Course:** GenAI Course  

### Problem Statement
Build a content-based recommendation system using TF-IDF and Cosine Similarity, create a Streamlit web app, and deploy on Render.

## Part 1: Load and Preprocess Data

In [ ]:
# Task 1 & 2: Load and clean data
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = pd.read_csv('movies.csv')
print('Dataset Shape:', df.shape)

# handle missing values and combine text
df['genres'] = df['genres'].fillna('')
df['overview'] = df['overview'].fillna('')
df['tags'] = df['genres'] + ' ' + df['overview']

# clean text
df['clean_text'] = df['tags'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True)
stop_words = set(ENGLISH_STOP_WORDS)
df['clean_text'] = df['clean_text'].apply(lambda x: ' '.join([w for w in x.split() if w not in stop_words]))

df.to_csv('cleaned_movies.csv', index=False)
print('Sample cleaned text:', df['clean_text'].iloc[0][:80], '...')

## Part 2: TF-IDF Vectorization & Cosine Similarity

In [ ]:
# Task 3 & 4: TF-IDF and Cosine Similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['clean_text'])
print('TF-IDF Shape:', tfidf_matrix.shape)

similarity = cosine_similarity(tfidf_matrix)
print('Cosine Similarity Shape:', similarity.shape)

## Part 3: Recommendation Function & Testing

In [ ]:
# Task 5: Recommendation function
def recommend(movie_name, top_n=5):
    idx = df[df['title'].str.lower() == movie_name.lower()].index[0]
    distances = list(enumerate(similarity[idx]))
    sorted_movies = sorted(distances, key=lambda x: x[1], reverse=True)[1:top_n+1]
    return [(df.iloc[i]['title'], df.iloc[i]['genres'], round(score, 4)) for i, score in sorted_movies]

for test_m in ['The Dark Knight', 'Inception', 'Toy Story']:
    print(f'\nRecommendations for {test_m}:')
    for title, genre, score in recommend(test_m):
        print(f'  - {title} ({genre}) | score: {score}')

## Part 4: Streamlit UI & Deployment
Run the app locally:
```bash
streamlit run app.py
```

### Render Deployment Steps
1. Push code to GitHub repository.
2. In Render dashboard, click **New + Web Service**.
3. Connect the repository.
4. Set Build Command: `pip install -r requirements.txt`
5. Set Start Command: `streamlit run app.py --server.port $PORT --server.address 0.0.0.0`
6. Deploy!